# OnioRakshak — Shelf-Life & Risk Model Training

This notebook trains the two models used in production:
1. A **regressor** predicting `remaining_shelf_life_days`
2. A **classifier** predicting a `Safe / Warning / Critical` risk status

It follows directly from `01_eda_and_preprocessing.ipynb` — the feature
choices here (which columns to use, why some are excluded, why the split
is batch-aware) were justified there. This notebook focuses on training,
evaluation, and saving the final model artifacts.

Trained models are saved to `../models/` as `.joblib` files, which
`src/appwrite_predict_function.py` loads for real-time inference.


## 1. Imports & Setup

Loading the libraries needed for preprocessing, model training, evaluation,
and saving the trained pipelines.


In [1]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    mean_absolute_error,
    r2_score,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

MODEL_DIR = "../models"
os.makedirs(MODEL_DIR, exist_ok=True)


## 2. Load Data

Loading the same dataset used in the EDA notebook.


In [2]:
df = pd.read_csv("../data/onion_shelf_life_dataset.csv")
df.shape


(21780, 21)

## 3. Define the Feature Set

**Why these features and not others:** only sensor-derivable readings and
one-time intake entries are used. `weight_loss_pct`, `decay_pct`, and
`sprouting_pct` are deliberately excluded — they're symptoms of spoilage
that our ESP32 + DHT22/MQ135 sensor rig cannot actually measure in the
field. Training on them would inflate offline metrics while producing a
model that's useless once wired to real hardware (this was confirmed in
the EDA notebook, where excluding them dropped R² from 0.95 to a more
honest 0.91).


In [3]:
NUMERIC_FEATURES = [
    "initial_quality_score",   # manual entry at intake (quality grading)
    "curing_score",            # manual entry at intake
    "temperature_C",
    "relative_humidity_pct",
    "mq135_gas_index",
    "avg_temperature_24h_C",
    "avg_humidity_24h_pct",
    "avg_mq135_24h",
    "hours_temp_above_27C_24h",
    "hours_RH_above_65pct_24h",
    "storage_day",             # known operationally (days since batch was stored)
]
CATEGORICAL_FEATURES = ["variety", "storage_mode"]
TARGET_REGRESSION = "remaining_shelf_life_days"


## 4. Create the Risk Status Label

**Why this matters:** the dashboard and actuator logic need a discrete,
glanceable status rather than a raw day-count. This buckets the regression
target into three categories. These thresholds (7 / 25 days) are a starting
point for the demo — revisit with the project guide once real
storage/spoilage data is available to validate against.


In [5]:
def make_risk_label(days: int) -> str:
    if days <= 7:
        return "Critical"
    if days <= 25:
        return "Warning"
    return "Safe"

df["risk_status"] = df[TARGET_REGRESSION].apply(make_risk_label)
df["risk_status"].value_counts()


risk_status
Safe        12286
Critical     6254
Warning      3240
Name: count, dtype: int64

## 5. Batch-Aware Train/Test Split

**Why this matters:** rows from the same `batch_id` share one decay
trajectory. A random row-level split would leak future information about a
batch into its own test set. Splitting by `batch_id` with
`GroupShuffleSplit` keeps every batch entirely in either train or test.


In [6]:
splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, test_idx = next(splitter.split(df, groups=df["batch_id"]))
train_df, test_df = df.iloc[train_idx], df.iloc[test_idx]

X_train = train_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
X_test = test_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y_train_reg = train_df[TARGET_REGRESSION]
y_test_reg = test_df[TARGET_REGRESSION]
y_train_clf = train_df["risk_status"]
y_test_clf = test_df["risk_status"]

print(f"Train: {train_df['batch_id'].nunique()} batches, {len(train_df)} rows")
print(f"Test:  {test_df['batch_id'].nunique()} batches, {len(test_df)} rows")


Train: 144 batches, 17424 rows
Test:  36 batches, 4356 rows


## 6. Build the Preprocessing Pipeline

**Why this matters:** bundling `StandardScaler` (numeric features) and
`OneHotEncoder` (categorical features) into a single `ColumnTransformer`
means the exact same preprocessing is guaranteed to run at inference time
as at training time — no risk of the Appwrite Function preprocessing data
slightly differently than training did.


In [7]:
def build_preprocessor() -> ColumnTransformer:
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), NUMERIC_FEATURES),
            ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
        ]
    )


## 7. Train the Shelf-Life Regressor

**Why Random Forest:** the relationship between gas/temperature/humidity
and spoilage is non-linear (confirmed in the EDA notebook — raw correlations
were often weak while threshold-based features mattered more). Random
Forest handles this without manual feature transforms, is robust to noisy
sensor data, and gives feature importances for free.

Hyperparameters (`max_depth=12`, `min_samples_leaf=10`, `n_estimators=100`)
are deliberately constrained rather than left at defaults. Uncapped trees on
this dataset produced a 300+MB model file for barely any accuracy gain —
since this model needs to be deployed inside an Appwrite Function with
package size limits, a small, near-equally-accurate model is the right
tradeoff over a marginally better but much heavier one.


In [8]:
reg_pipeline = Pipeline(steps=[
    ("preprocess", build_preprocessor()),
    ("model", RandomForestRegressor(n_estimators=100, max_depth=12, min_samples_leaf=10, random_state=42, n_jobs=-1)),
])
reg_pipeline.fit(X_train, y_train_reg)
reg_preds = reg_pipeline.predict(X_test)

print(f"MAE: {mean_absolute_error(y_test_reg, reg_preds):.2f} days")
print(f"R^2: {r2_score(y_test_reg, reg_preds):.3f}")


MAE: 6.43 days
R^2: 0.907


### Significance of this result

An MAE of ~6 days means the model's shelf-life estimate is typically off by
under a week — solid for a first working version, and good enough to drive
"ventilate now" style decisions. The R² (~0.91) shows the model explains
most of the variance in shelf-life using only sensor-realistic inputs.


## 8. Train the Risk Classifier

**Why Random Forest Classifier (not Logistic Regression):** the boundaries
between Safe/Warning/Critical aren't linear in the sensor readings — a
linear model would systematically misclassify the transition zones. Random
Forest's non-linear decision boundaries handle this better.


In [9]:
clf_pipeline = Pipeline(steps=[
    ("preprocess", build_preprocessor()),
    ("model", RandomForestClassifier(n_estimators=100, max_depth=12, min_samples_leaf=10, random_state=42, n_jobs=-1)),
])
clf_pipeline.fit(X_train, y_train_clf)
clf_preds = clf_pipeline.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test_clf, clf_preds):.3f}")
print(classification_report(y_test_clf, clf_preds))


Accuracy: 0.935
              precision    recall  f1-score   support

    Critical       0.91      0.96      0.94      1309
        Safe       0.97      0.98      0.98      2399
     Warning       0.83      0.70      0.76       648

    accuracy                           0.93      4356
   macro avg       0.90      0.88      0.89      4356
weighted avg       0.93      0.93      0.93      4356



### Significance of this result

Accuracy above 90% means the classifier reliably backs the dashboard/buzzer
logic. The "Warning" class is expected to have the weakest recall — it's
the narrow middle band between two easier-to-separate extremes (Safe and
Critical), so some blur at that boundary is inherent to the problem, not a
modeling flaw.


## 9. Feature Importance

**Why this matters:** beyond just accuracy, knowing *which* sensor drives
the prediction most is directly useful for the IEEE paper's analysis and
for prioritizing sensor reliability in the hardware design.


In [10]:
ohe = reg_pipeline.named_steps["preprocess"].named_transformers_["cat"]
feature_names = NUMERIC_FEATURES + list(ohe.get_feature_names_out(CATEGORICAL_FEATURES))
importances = reg_pipeline.named_steps["model"].feature_importances_

importance_df = pd.DataFrame({"feature": feature_names, "importance": importances})
importance_df.sort_values("importance", ascending=False).head(8)


,feature,importance
4,mq135_gas_index,0.730676
10,storage_day,0.172464
1,curing_score,0.024747
0,initial_quality_score,0.016068
2,temperature_C,0.007928
16,storage_mode_Warm_ambient,0.007540
5,avg_temperature_24h_C,0.007370
3,relative_humidity_pct,0.006157


**Reading this:** consistent with the EDA notebook, `mq135_gas_index` and
`storage_day` dominate. This validates the gas sensor as the most
operationally important sensor in the whole system.


## 10. Save the Trained Models

**Why this matters:** saving the full pipeline (preprocessing + model
together) means the Appwrite Function doesn't need to reimplement any
preprocessing logic — it just loads the pipeline and calls `.predict()` on
raw feature values.

`compress=3` is passed to `joblib.dump` to keep the saved file size down
further — helpful given the Appwrite Function deployment size constraint.


In [11]:
joblib.dump(reg_pipeline, f"{MODEL_DIR}/shelf_life_regressor.joblib", compress=3)
joblib.dump(clf_pipeline, f"{MODEL_DIR}/risk_classifier.joblib", compress=3)
print(f"Saved models to {MODEL_DIR}/")


Saved models to ../models/


## 11. Summary

- Shelf-life regressor: **MAE ≈ 6.4 days, R² ≈ 0.91**, trained only on
  sensor-realistic features
- Risk classifier: **~94% accuracy**, weakest on the "Warning" middle class
- Most predictive feature: `mq135_gas_index`, followed by `storage_day`
- Both models saved as `.joblib` pipelines in `../models/`, ready for
  `src/appwrite_predict_function.py` to load
- **Next step:** retrain on real ESP32 sensor logs once hardware is
  deployed, and revisit the risk thresholds with the project guide
